# Ask the Cell — the agent

A grounded orchestrator: macro questions -> multi-hop, cited, confidence-scored answers. Every fact comes from a model layer (DepMap, gnomAD, DGIdb, network, kinetics, ...); nothing is invented.

Try: *'is MCL1 a good target'*, *'what happens if I knock out TP53'*, *'should we target EGFR'*, or feed a diseased-cell signature to *'where does this disease live'*.

No GPU needed.

In [ ]:
# setup (auto-restore cell_complete.json from Drive)
import subprocess, os, sys, shutil, glob
if not os.path.isdir('/content/cell'):
    subprocess.run('git clone -b claude/vectorize-gex-propensity-NRqBW https://github.com/nikku03/cell.git /content/cell', shell=True)
os.chdir('/content/cell'); subprocess.run(['git','pull']); subprocess.run('pip install -q scipy scikit-learn pandas', shell=True)
os.makedirs('outputs/orphan', exist_ok=True)
if not os.path.exists('outputs/orphan/cell_complete.json'):
    try:
        from google.colab import drive; drive.mount('/content/drive')
        h=[x for x in glob.glob('/content/drive/MyDrive/**/cell_complete.json',recursive=True) if os.path.getsize(x)>1e5]
        if h: shutil.copy(h[0],'outputs/orphan/'); print('restored cell_complete.json')
    except Exception as e: print('restore note:',e)
sys.path.insert(0,'colab'); from cell_agent import CellAgent; agent=CellAgent(); print('agent ready')

In [ ]:
# ask anything (edit the question)
print(agent.ask('is MCL1 a good target to develop'))

In [ ]:
# a few more
for q in ['what happens if I knock out TP53','should we target KRAS','is BRCA1 druggable']:
    print('>>>',q); print(agent.ask(q)); print()

In [ ]:
# reverse inference: give a diseased-cell signature -> where does the disease live + dossier
# (example: build a signature from a driver's targets; in practice use your diseased-vs-healthy profile)
from compute_reverse_inference import load_model
M=load_model(); G=M['G']; reg=M['regulon']
seed=next(i for i in reg if G[i]['name']=='MYC' and len(reg[i])>=20)
sig={G[t]['name']: sg*2.0 for t,sg in reg[seed]}
print(agent.ask('where does this disease live', signature=sig))